In [1]:
import sqlite3

conn = sqlite3.connect("law.db")
cursor = conn.cursor()

def extract_from_db(id: int, include_parent=False):
    cursor.execute("SELECT * FROM laws WHERE id = ?", (id,))
    rows = cursor.fetchall()

    columns = [col[0] for col in cursor.description]
    results = [dict(zip(columns, row)) for row in rows]

    if include_parent:
        all_nodes = []
        for r in results:
            current = r
            while current['parent_id'] is not None:
                cursor.execute("SELECT * FROM laws WHERE id = ?", (current['parent_id'],))
                parent = cursor.fetchone()
                if parent is None:
                    break
                parent_dict = dict(zip(columns, parent))
                all_nodes.append(parent_dict)
                current = parent_dict
        results.extend(all_nodes)

    return results[::-1]

In [2]:
import re

pos_map = {
    "R": "Adverb (Trạng từ)",
    "V": "Verb (Động từ)",
    "N": "Noun (Danh từ)",
    "E": "Preposition (Giới từ)",
    "P": "Pronoun (Đại từ)",
    "CH": "Punctuation (Dấu câu)",
    "A": "Adjective (Tính từ)",
    "M": "Numeral (Số từ)"
}

if "rdrsegmenter" not in globals():
    import py_vncorenlp
    rdrsegmenter = py_vncorenlp.VnCoreNLP(
        annotators=["wseg", "pos"],
        save_dir=r"E:\Github\LawAssistant\triplet_extraction\VnCoreNLP-master"
    )

def clean_text(text: str) -> str:
    """Remove newlines, tabs, extra spaces, punctuation and lowercase."""
    text = re.sub(r"[\r\n\t]+", " ", text)
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"[,.!?;:]+", "", text)
    return text.strip().lower()

def process_sentence(text: str, rdrsegmenter, verbose: bool = True) -> dict:
    """
    Process a Vietnamese sentence and print results along the way.
    Extracts: Cleaned text, Segmentation, POS, Important tokens (N, V, A).
    """

    results = {
        "original": text,
        "cleaned": "",
        "segmented": [],
        "pos_annotation": [],
        "concepts": []
    }

    if verbose:
        print("1. Original text:")
        print(text, "\n")

    # 2. Clean
    text = clean_text(text)
    results["cleaned"] = text
    if verbose:
        print("2. Cleaned text:")
        print(text, "\n")

    # 3. Segmentation
    segmented = rdrsegmenter.word_segment(text)
    results["segmented"] = segmented
    if verbose:
        print("3. Segmented text (tokens):")
        print(segmented, "\n")

    # 4. POS tagging
    output = rdrsegmenter.annotate_text(text)
    sents = output.values() if isinstance(output, dict) else output

    pos_annot = []
    important_tokens = []  # flat list
    keep_tags = ("N", "A", "V")

    if verbose:
        print("4. POS annotation:")
        print(f"{'Idx':<5} {'Token':<15} {'POS':<20}")
        print("-" * 45)

    for sent in sents:
        if not isinstance(sent, list):
            continue
        for token in sent:
            if not isinstance(token, dict):
                continue
            word = token.get("wordForm", "")
            pos = token.get("posTag", "")
            pos_full = pos_map.get(pos, pos)

            pos_data = {
                "index": token.get("index", ""),
                "token": word,
                "pos": pos_full
            }
            pos_annot.append(pos_data)

            if verbose:
                print(f"{pos_data['index']:<5} {pos_data['token']:<15} {pos_data['pos']:<20}")

            # collect important tokens
            if any(pos.startswith(tag) for tag in keep_tags):
                important_tokens.append(word)

    results["pos_annotation"] = pos_annot
    results["concepts"] = important_tokens

    if verbose:
        print("\n5. Extracted important tokens (N, V, A):")
        print(important_tokens, "\n")

    return results

In [3]:
from openai import OpenAI
from dotenv import load_dotenv
import os

# Load .env file
load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")

def get_gpt_response(user_prompt, system_prompt, api_key, model="gpt-4o-mini", max_token=500):
    client = OpenAI(api_key=api_key)
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        max_tokens=max_token
    )
    print(response.choices[0].message.content)
    return response.choices[0].message.content

In [4]:
rows = extract_from_db(629, include_parent=True)
law = ""
title = ""
for r in rows:
    title += r['title'] + " "
    if not (r['title'].strip().startswith("Chương") or  r['title'].strip().startswith("Điều")):
        law += r['content'] + "\n"

so_hieu = r['so_hieu'] 
print(title)
print(clean_text(law))

Chương IV Điều 57 Khoản 1 Điểm m 
giấy phép lái xe bao gồm các hạng sau đây hạng ce cấp cho người lái các loại xe ô tô quy định cho giấy phép lái xe hạng c kéo rơ moóc có khối lượng toàn bộ theo thiết kế trên 750 kg xe ô tô đầu kéo kéo sơ mi rơ moóc


In [7]:
system_prompt_rewrite = """
Bạn là trợ lý AI Tiếng Việt chuyên nghiệp và trung thực.
Bạn là chuyên gia pháp luật Việt Nam, am hiểu các bộ luật, nghị định, và văn bản pháp luật.
Bạn là chuyên gia ngôn ngữ Việt Nam, biết viết câu chuẩn cấu trúc, chính xác, trang trọng, và đúng ngôn ngữ pháp lý.
Luôn trả lời chính xác, hữu ích, ngắn gọn và an toàn.
Nếu thông tin không hợp lý hoặc thiếu, hãy yêu cầu thêm thông tin thay vì đoán mò.
Không thay đổi ý nghĩa khi viết lại câu.
Luôn dùng ngôn ngữ chính xác như trong văn bản pháp luật, tránh ngôn ngữ thông thường hay không trang trọng.
"""

user_prompt_rewrite =f"""
Ngữ cảnh: Bộ luật số {so_hieu} trong luật Việt Nam
Nhiệm vụ: Viết lại câu sau để hoàn chỉnh cấu trúc với đầy đủ chủ ngữ, vị ngữ, giữ nguyên ý nghĩa.
Câu cần viết lại: "{clean_text(law)}"
"""

In [8]:
print(user_prompt_rewrite)


Ngữ cảnh: Bộ luật số 36/2024/QH15 trong luật Việt Nam
Nhiệm vụ: Viết lại câu sau để hoàn chỉnh cấu trúc với đầy đủ chủ ngữ, vị ngữ, giữ nguyên ý nghĩa.
Câu cần viết lại: "giấy phép lái xe bao gồm các hạng sau đây hạng ce cấp cho người lái các loại xe ô tô quy định cho giấy phép lái xe hạng c kéo rơ moóc có khối lượng toàn bộ theo thiết kế trên 750 kg xe ô tô đầu kéo kéo sơ mi rơ moóc"



In [23]:
rewrite_sentence = get_gpt_response(user_prompt_rewrite, system_prompt_rewrite, api_key, "gpt-4")

"Giấy phép lái xe được chia thành nhiều hạng, trong đó có hạng CE được cấp cho người lái các loại xe ô tô quy định dành cho giấy phép lái xe hạng C. Đối tượng của hạng CE là những chiếc xe có khả năng kéo rơ moóc với khối lượng toàn bộ theo thiết kế trên 750kg, bao gồm xe ô tô đầu kéo và xe sơ mi rơ moóc."


In [9]:
rewrite_sentence = """
"Giấy phép lái xe được chia thành nhiều hạng, trong đó có hạng CE được cấp cho người lái các loại xe ô tô quy định dành cho giấy phép lái xe hạng C. Đối tượng của hạng CE là những chiếc xe có khả năng kéo rơ moóc với khối lượng toàn bộ theo thiết kế trên 750kg, bao gồm xe ô tô đầu kéo và xe sơ mi rơ moóc."
"""

In [10]:
res = process_sentence(rewrite_sentence, rdrsegmenter, verbose=True)
keywords = res['concepts']

1. Original text:

"Giấy phép lái xe được chia thành nhiều hạng, trong đó có hạng CE được cấp cho người lái các loại xe ô tô quy định dành cho giấy phép lái xe hạng C. Đối tượng của hạng CE là những chiếc xe có khả năng kéo rơ moóc với khối lượng toàn bộ theo thiết kế trên 750kg, bao gồm xe ô tô đầu kéo và xe sơ mi rơ moóc."
 

2. Cleaned text:
"giấy phép lái xe được chia thành nhiều hạng trong đó có hạng ce được cấp cho người lái các loại xe ô tô quy định dành cho giấy phép lái xe hạng c đối tượng của hạng ce là những chiếc xe có khả năng kéo rơ moóc với khối lượng toàn bộ theo thiết kế trên 750kg bao gồm xe ô tô đầu kéo và xe sơ mi rơ moóc" 

3. Segmented text (tokens):
['" giấy_phép lái_xe được chia thành nhiều hạng trong đó có_hạng ce được cấp cho người lái các loại xe ô_tô quy_định dành cho giấy_phép lái_xe hạng c đối_tượng của hạng ce là những chiếc xe có khả_năng kéo rơ moóc với khối_lượng toàn_bộ theo thiết_kế trên 750kg bao_gồm xe ô_tô đầu kéo và xe sơ_mi rơ moóc "'] 

4. POS 

In [11]:
system_prompt_triplet = """
Bạn là một trợ lý AI chuyên về phân tích văn bản pháp luật Việt Nam.
Nhiệm vụ: trích xuất các thực thể (entities) và quan hệ (relations) từ văn bản pháp luật.

Định nghĩa và quy tắc chung:
1. Thực thể (Concept): danh từ hoặc cụm danh từ, hoặc tính từ kết hợp với danh từ — đại diện cho đối tượng, sự vật, khái niệm trong văn bản pháp luật.
2. Quan hệ (Relation):
   - Là động từ hoặc cụm động từ mô tả hành động, trạng thái hoặc mối liên hệ giữa hai thực thể.
   - Giữ cả cụm nếu động từ kết hợp với giới từ hoặc bổ ngữ quan trọng cho nghĩa pháp luật.
   - Luôn nằm giữa Concept1 và Concept2 trong triplet.
3. Mỗi triplet có định dạng: ["Concept1", "Relation", "Concept2"].
4. Mỗi thực thể và quan hệ phải là đơn vị nhỏ nhất có thể chia được nhưng vẫn bảo toàn ý nghĩa pháp lý.
5. Không được thêm, sửa, hoặc thay thế bất kỳ từ/ký tự nào ngoài token đã cung cấp.
6. Nếu văn bản không đủ thông tin để tạo triplet theo quy tắc, trả về ["Thiếu thông tin"].
7. Tránh giải thích, bổ sung, hoặc suy đoán ngoài văn bản.
8. Trả lời chỉ dưới dạng một danh sách JSON các triplet — không kèm chú thích, chữ ngoài JSON, hoặc dòng trạng thái.
9. Được phép ghép các token gần nhau. Luôn giữ thứ tự xuất hiện ban đầu của các token khi ghép thành thực thể lớn hơn (chỉ được ghép các token kề nhau).
10. Bỏ đi các thành phần câu như trạng từ, giới từ, liên từ, thán từ, phó từ.
"""

user_prompt_triplet = f"""
Ngữ cảnh: Bộ luật số {so_hieu} trong luật Việt Nam
Câu gốc: {rewrite_sentence}
Từ đã được trích xuất (giữ nguyên, có dấu gạch dưới): {keywords}

Nhiệm vụ: Tạo danh sách các triplet ở dạng ["Concept1", "Relation", "Concept2"] chỉ bằng các token và dựa trên câu gốc. Có thể sử dụng một concept nhiều lần nhưng không được thay đổi ý nghĩa của câu.
"""

print(system_prompt_triplet)
print(user_prompt_triplet)


Bạn là một trợ lý AI chuyên về phân tích văn bản pháp luật Việt Nam.
Nhiệm vụ: trích xuất các thực thể (entities) và quan hệ (relations) từ văn bản pháp luật.
Bỏ đi các thành phần câu như trạng từ, giới từ, liên từ, thán từ, phó từ.

Định nghĩa và quy tắc chung:
1. Thực thể (Concept): danh từ hoặc cụm danh từ, hoặc tính từ kết hợp với danh từ — đại diện cho đối tượng, sự vật, khái niệm trong văn bản pháp luật.
2. Quan hệ (Relation):
   - Là động từ hoặc cụm động từ mô tả hành động, trạng thái hoặc mối liên hệ giữa hai thực thể.
   - Giữ cả cụm nếu động từ kết hợp với giới từ hoặc bổ ngữ quan trọng cho nghĩa pháp luật.
   - Luôn nằm giữa Concept1 và Concept2 trong triplet.
3. Mỗi triplet có định dạng: ["Concept1", "Relation", "Concept2"].
4. Mỗi thực thể và quan hệ phải là đơn vị nhỏ nhất có thể chia được nhưng vẫn bảo toàn ý nghĩa pháp lý.
5. Không được thêm, sửa, hoặc thay thế bất kỳ từ/ký tự nào ngoài token đã cung cấp.
6. Nếu văn bản không đủ thông tin để tạo triplet theo quy tắc, 

In [12]:
triplet = get_gpt_response(user_prompt_triplet, system_prompt_triplet, api_key, "gpt-4")

[
["Giấy phép lái xe", "được chia thành", "nhiều hạng"],
["hạng CE", "được cấp cho", "người lái loại xe ô tô"],
["loại xe ô tô", "quy định dành cho", "giấy phép lái xe hạng C"],
["đối tượng của hạng CE", "là", "chiếc xe có khả năng kéo rơ moóc"],
["chiếc xe có khả năng kéo rơ moóc", "với", "khối lượng toàn bộ theo thiết kế trên 750kg"],
["chiếc xe có khả năng kéo rơ moóc", "bao gồm", "xe ô tô đầu kéo và xe sơ mi rơ moóc"]
]
